# Separando Páginas e Criando PDFs <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg" height="45" /> ✂️

<img src="https://img.shields.io/badge/Jupyter-111827?style=flat-square&logo=jupyter&logoColor=F37626" alt="Jupyter" />
<img src="https://img.shields.io/badge/Python-111827?style=flat-square&logo=python&logoColor=3776AB" alt="Python" />
<img src="https://img.shields.io/badge/PDF-EC1C24?style=flat-square&logo=adobeacrobatreader&logoColor=white" alt="PDF" />
<img src="https://img.shields.io/badge/python-3.14+-blue" alt="Python Version" />
<img src="https://img.shields.io/badge/tópico-pdf%20%7C%20páginas%20%7C%20tabelas-teal" alt="Tópico" />
<img src="https://img.shields.io/badge/dificuldade-Intermediário-yellow" alt="Dificuldade" />
<img src="https://img.shields.io/badge/pré--req-strings%20%7C%20listas%20%7C%20funções%20%7C%20pandas-purple" alt="Pré-req" />
<img src="https://img.shields.io/badge/requer-pip%20install%20pypdf%20%7C%20tabula--py%20%7C%20pikepdf-orange" alt="Biblioteca" />

> Separar as páginas certas de um relatório gigante, juntar dois PDFs num só, girar uma página escaneada de lado, ler o DRE direto de dentro de uma tabela... o `pypdf` cuida da manipulação do arquivo (juntar, separar, girar, proteger), o `tabula-py` lê tabelas bem estruturadas como `DataFrame`, e o `pikepdf` chega nas imagens embutidas no PDF. Como no restante do módulo, usamos os relatórios trimestrais do Magalu (3T20 e 4T20) como material de exemplo.

## 📋 Conteúdo

1. [Separando um PDF em Várias Páginas](#-1-separando-um-pdf-em-várias-páginas)
2. [Juntando Páginas Específicas em um Novo PDF](#-2-juntando-páginas-específicas-em-um-novo-pdf)
3. [Juntando PDFs Inteiros (Mesclando)](#-3-juntando-pdfs-inteiros-mesclando)
4. [Inserindo uma Página no Meio de Outro PDF](#-4-inserindo-uma-página-no-meio-de-outro-pdf)
5. [Girando Páginas de um PDF](#-5-girando-páginas-de-um-pdf)
6. [Localizando um Texto Dentro do PDF](#-6-localizando-um-texto-dentro-do-pdf)
7. [Extraindo Tabelas com o Tabula](#-7-extraindo-tabelas-com-o-tabula)
8. [Tabelas em Páginas com Mais de uma Tabela](#-8-tabelas-em-páginas-com-mais-de-uma-tabela)
9. [Extraindo Imagens de Dentro do PDF](#-9-extraindo-imagens-de-dentro-do-pdf)
10. [Substituindo Texto num PDF (Nota)](#-10-substituindo-texto-num-pdf-nota)


Para instalar 🕹️:

```
pip install pypdf tabula-py pikepdf
```

> ⚠️ **Atenção:** `tabula-py` depende do Java (JDK 8+) instalado na máquina — sem ele, `tabula.read_pdf` falha na primeira chamada.


## ✂️ 1. Separando um PDF em Várias Páginas

Pra mandar só a página que interessa (tipo o DRE pra Diretoria) sem enviar o relatório inteiro, dá pra quebrar o PDF em um arquivo por página: um `PdfWriter` novo pra cada página do `PdfReader` original.

| Método 🔑 | O que faz 🔓 |
|---|---|
| `PdfReader(caminho)` | abre um PDF existente pra leitura |
| `PdfWriter()` | cria um PDF novo, vazio, pronto pra receber páginas |
| `writer.add_page(pagina)` | adiciona uma única página ao `writer` |
| `writer.write(caminho)` | grava o PDF montado no `writer` num arquivo `.pdf` |

> 💡 **Dica:** `enumerate(reader.pages)` traz o índice de cada página junto — soma-se `+ 1` pra nomear o arquivo já a partir da página 1, já que o índice da lista começa em 0.


In [1]:
import os
from pypdf import PdfReader, PdfWriter
from cores import *

caminho_3t20 = "../.spec/Integração Python e PDF - Como funciona - MGLU_ER_3T20_POR.pdf"
pasta_paginas_mglu = "../.spec/MGLU_3T20_Paginas"

os.makedirs(pasta_paginas_mglu, exist_ok=True)

reader = PdfReader(caminho_3t20)

for indice, pagina in enumerate(reader.pages):
    writer = PdfWriter()
    writer.add_page(pagina)
    with open(f"{pasta_paginas_mglu}/Pagina {indice + 1}.pdf", "wb") as arquivo:
        writer.write(arquivo)

print(f"{CinzaClaro}Páginas geradas:{Reset} {MagentaClaro}{len(os.listdir(pasta_paginas_mglu))}{Reset}")


Páginas geradas: 24


## 🧩 2. Juntando Páginas Específicas em um Novo PDF

Com o relatório já separado página por página, dá pra montar um novo PDF só com as páginas que interessam — Destaques, DRE e Balanço, por exemplo — juntando-as num único `PdfWriter`, na ordem desejada.

| Método 🔑 | O que faz 🔓 |
|---|---|
| `PdfReader(f"...{n}.pdf")` | reabre o arquivo de uma única página específica |
| `writer.add_page(reader.pages[0])` | adiciona a única página desse `PdfReader` ao PDF final |

> 💡 **Dica:** como cada página separada virou um PDF de 1 página só, o índice usado é sempre `[0]` — é sempre a primeira (e única) página daquele arquivo.


In [2]:
paginas_desejadas = [1, 14, 16]  # Destaques, DRE e Balanço
writer_consolidado = PdfWriter()

for pagina in paginas_desejadas:
    reader_pagina = PdfReader(f"{pasta_paginas_mglu}/Pagina {pagina}.pdf")
    writer_consolidado.add_page(reader_pagina.pages[0])

with open("../.spec/MGLU_3T20_Consolidado.pdf", "wb") as arquivo:
    writer_consolidado.write(arquivo)

print(f"{CinzaClaro}Páginas no consolidado:{Reset} {MagentaClaro}{len(PdfReader('../.spec/MGLU_3T20_Consolidado.pdf').pages)}{Reset}")


Páginas no consolidado: 3


## 📎 3. Juntando PDFs Inteiros (Mesclando)

Quando o objetivo é juntar 2 relatórios inteiros — não só páginas soltas —, o `writer.append()` resolve isso de uma vez, sem precisar percorrer página por página.

| Método 🔑 | O que faz 🔓 |
|---|---|
| `PdfWriter()` | cria um PDF novo, vazio |
| `writer.append(caminho)` | adiciona **todas** as páginas de um PDF ao final do `writer` |
| `writer.write(caminho)` | grava o PDF montado num arquivo `.pdf` |

> 💡 **Dica:** dá pra chamar `writer.append()` várias vezes seguidas — cada chamada emenda mais um PDF inteiro no final do que já foi montado.


In [3]:
caminho_4t20 = "../.spec/Integração Python e PDF - Como funciona - MGLU_ER_4T20_POR.pdf"

writer_mesclado = PdfWriter()
writer_mesclado.append(caminho_3t20)
writer_mesclado.append(caminho_4t20)

with open("../.spec/MGLU_Mesclado.pdf", "wb") as arquivo:
    writer_mesclado.write(arquivo)

print(f"{CinzaClaro}Páginas no mesclado:{Reset} {MagentaClaro}{len(PdfReader('../.spec/MGLU_Mesclado.pdf').pages)}{Reset}")


Páginas no mesclado: 49


## ➕ 4. Inserindo uma Página no Meio de Outro PDF

Às vezes não dá pra só emendar no final — por exemplo, colocar a página de Destaques do 3T20 logo depois da capa do relatório do 4T20, pra comparar os 2 trimestres num único arquivo. O `merge()` faz isso, recebendo a posição onde a página entra.

| Método 🔑 | O que faz 🔓 |
|---|---|
| `writer.merge(posicao, caminho)` | insere as páginas de outro PDF numa posição específica do `writer`, empurrando o resto pra frente |

> ⚠️ **Atenção:** `posicao` é o índice (começa em 0) de onde a página vai entrar, não o número final da página no relatório — `merge(1, ...)` insere logo depois da 1ª página (índice 0).


In [4]:
writer_insercao = PdfWriter()
writer_insercao.append(caminho_4t20)
writer_insercao.merge(1, f"{pasta_paginas_mglu}/Pagina 1.pdf")

with open("../.spec/MGLU_4T20_com_Destaques_3T20.pdf", "wb") as arquivo:
    writer_insercao.write(arquivo)

print(f"{CinzaClaro}Páginas no 4T20 com a página inserida:{Reset} {MagentaClaro}{len(PdfReader('../.spec/MGLU_4T20_com_Destaques_3T20.pdf').pages)}{Reset}")


Páginas no 4T20 com a página inserida: 26


## 🔄 5. Girando Páginas de um PDF

Cada página é um objeto que pode ser girado antes de ser adicionado ao `writer` — útil pra corrigir um PDF escaneado de lado, por exemplo.

| Método 🔑 | O que faz 🔓 |
|---|---|
| `pagina.rotate(graus)` | gira a página em graus (múltiplos de 90), sentido horário |
| `writer.add_page(pagina)` | adiciona a página já girada ao `writer` |

> 💡 **Dica:** `rotate()` altera a própria página — não precisa reatribuir o retorno, só chamar e depois adicionar essa mesma página ao `writer`.


In [5]:
reader_rotacionar = PdfReader(caminho_3t20)
writer_rotacionado = PdfWriter()

for pagina in reader_rotacionar.pages:
    pagina.rotate(90)
    writer_rotacionado.add_page(pagina)

with open("../.spec/MGLU_3T20_Rotacionado.pdf", "wb") as arquivo:
    writer_rotacionado.write(arquivo)

print(f"{CinzaClaro}Rotação da 1ª página:{Reset} {MagentaClaro}{writer_rotacionado.pages[0].rotation}°{Reset}")


Rotação da 1ª página: 90°


## 🔎 6. Localizando um Texto Dentro do PDF

Dá pra procurar um trecho específico de texto (ex.: uma linha de despesas) sem abrir o PDF manualmente — percorre-se cada página com `extract_text()` e testa se o texto de referência está contido nela.

| Método 🔑 | O que faz 🔓 |
|---|---|
| `pagina.extract_text()` | devolve todo o texto da página como uma única string |
| `texto.find(referencia)` | devolve a posição onde o texto de referência começa (ou `-1` se não achar) |
| `texto[inicio:fim]` | recorta só o trecho de interesse a partir das posições encontradas |

> 💡 **Dica:** `texto.find("\n", posicao_inicial)` acha a próxima quebra de linha depois do início, isolando só aquela linha inteira do texto.


In [6]:
reader_texto = PdfReader(caminho_3t20)
texto_referencia = "Despesas com Vendas"

for indice, pagina in enumerate(reader_texto.pages):
    texto_pagina = pagina.extract_text()
    if texto_referencia in texto_pagina:
        print(f"{CinzaClaro}Encontrado na página:{Reset} {MagentaClaro}{indice + 1}{Reset}")
        posicao_inicial = texto_pagina.find(texto_referencia)
        posicao_final = texto_pagina.find("\n", posicao_inicial)
        print(f"{CinzaClaro}{Reset}{MagentaClaro}{texto_pagina[posicao_inicial:posicao_final]}{Reset}")
        break


Encontrado na página: 6
Despesas com Vendas          (1.432,6) -17,2%                                          -      (1.432,6) -17,2% 


## 📊 7. Extraindo Tabelas com o Tabula

Pra tabelas bem estruturadas (tipo o DRE), o `tabula-py` costuma reconhecer linhas e colunas melhor que a extração de texto puro — ele devolve uma lista de `DataFrame`s do pandas, um por tabela encontrada na página.

| Método 🔑 | O que faz 🔓 |
|---|---|
| `tabula.read_pdf(caminho, pages=n)` | lê a página `n` (numeração real do PDF, começando em 1) e devolve uma lista de tabelas |
| `df.dropna(how="all", axis=0)` | remove linhas totalmente vazias |
| `df.dropna(how="all", axis=1)` | remove colunas totalmente vazias |
| `df.columns = df.iloc[0]` | usa a primeira linha da tabela extraída como cabeçalho |

> ⚠️ **Atenção:** não confundir com o pacote `tabula` (uma calculadora de acordes musicais) — o certo é instalar `tabula-py`.


In [7]:
import tabula

tabelas_dre = tabula.read_pdf(caminho_3t20, pages=5)
df_dre = tabelas_dre[0]

# excluindo linhas e colunas totalmente vazias
df_dre = df_dre.dropna(how="all", axis=0)
df_dre = df_dre.dropna(how="all", axis=1)

# fixando a primeira linha como cabeçalho e resetando o índice
df_dre.columns = df_dre.iloc[0]
df_dre = df_dre.iloc[1:]
df_dre = df_dre.reset_index(drop=True)

display(df_dre)


Failed to import jpype dependencies. Fallback to subprocess.


No module named 'jpype'


,R$ milhões (exceto quando indicado),3T2,3T1,Var(%,9M2,9M1,Var(%
0,Vendas Totais1 (incluindo marketplace),"12.355,5","6.817,6","81,2","28.584,","18.282,6","56,3"
1,Receita Bruta,"10.349,5","5.999,4","72,5","23.652,","16.508,8","43,3"
2,Receita Líquida,"8.308,3","4.864,2","70,8","19.111,","13.501,3","41,6"
3,Lucro Bruto,"2.178,7","1.424,9","52,9","5.034,","3.728,6","35,0"
4,Margem Bruta,"26,2","29,3","-3,1 p","26,3","27,6","-1,3 p"
5,EBITDA,"546,1","501,2","9,0","1.022,","1.276,5","-19,9"
6,Margem EBITDA,"6,6","10,3","-3,7 p","5,3","9,5","-4,2 p"
7,Lucro Líquido,"206,0","235,1","-12,4","172,","753,8","-77,2"
8,Margem Líquida,"2,5","4,8","-2,3 p","0,9","5,6","-4,7 p"
9,Lucro Bruto - Ajustado,"2.178,7","1.488,9","46,3","5.034,","3.964,6","27,0"


## 🗂️ 8. Tabelas em Páginas com Mais de uma Tabela

Algumas páginas trazem 2 (ou mais) tabelas — `read_pdf` sempre devolve uma lista, então quando há mais de uma tabela, cada posição da lista é uma delas, na ordem em que aparecem na página.

| Método 🔑 | O que faz 🔓 |
|---|---|
| `len(tabelas)` | quantidade de tabelas encontradas na página |
| `for tabela in tabelas:` | percorre cada tabela separadamente pra tratar cada uma |

> 💡 **Dica:** dá pra desempacotar a lista direto (`tabela_a, tabela_b = tabelas`) quando já se sabe de antemão quantas tabelas a página tem.


In [8]:
tabelas_pagina12 = tabula.read_pdf(caminho_3t20, pages=12)
print(f"{CinzaClaro}Tabelas encontradas na página 12:{Reset} {MagentaClaro}{len(tabelas_pagina12)}{Reset}")

df_capital_giro, df_investimentos = tabelas_pagina12
df_capital_giro = df_capital_giro.dropna(how="all", axis=0)
df_investimentos = df_investimentos.dropna(how="all", axis=0)

display(df_capital_giro)
display(df_investimentos)


Tabelas encontradas na página 12: 2


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,(+) Contas a Receber (sem Cartões de Crédito),"(26,7)","706,3","680,8","781,3","794,0","733,0"
1,(+) Estoques,"2.120,2","5.005,9","4.198,2","4.075,5","3.801,8","2.885,7"
2,(+) Partes Relacionadas (sem Cartão Luiza),"(10,5)","71,3","80,4","77,1","100,6","81,8"
3,(+) Impostos a Recuperar,"186,3","932,0","748,9","877,4","864,1","745,7"
4,(+) Outros Ativos,"(56,6)","88,5","100,2","143,5","136,3","145,1"
5,(+) Ativos Circulantes Operacionais,"2.212,7","6.804,0","5.808,6","5.954,8","5.696,8","4.591,3"
7,(-) Fornecedores,"2.301,5","6.104,3","5.334,0","4.132,7","5.934,9","3.802,8"
8,(-) Repasses e outros depósitos,"627,3","627,3","639,3","235,9",-,-
9,"(-) Salários, Férias e Encargos Sociais","95,0","444,7","329,0","263,3","354,7","349,8"
10,(-) Impostos a Recolher,"90,7","299,6","206,4","176,9","352,0","208,8"


,R$ milhões,3T20,%,3T19,%.1,Var(%),9M20,%.2,9M20.1,%.3,Var(%).1
1,Lojas Novas,"21,2",14%,"94,6",51%,-78%,"69,0",21%,"121,9",31%,-43%
2,Reformas,"6,1",4%,"8,2",4%,-26%,"14,6",4%,"38,2",10%,-62%
3,Tecnologia,"69,1",45%,"32,6",18%,112%,"147,6",45%,"84,4",22%,75%
4,Logística,"36,3",24%,"32,8",18%,11%,"62,1",19%,"107,4",27%,-42%
5,Outros,"21,5",14%,"17,7",10%,22%,"35,7",11%,"38,8",10%,-8%
6,Total,"154,2",100%,"186,0",100%,-17%,"329,1",100%,"390,7",100%,-16%


## 🖼️ 9. Extraindo Imagens de Dentro do PDF

Pra manipular objetos que não são texto (imagens, ex.: um ícone ou logo), o `pikepdf` acessa diretamente os `XObjects` da página — diferente do `pypdf`/`tabula-py`, que trabalham com página e texto/tabela.

| Método 🔑 | O que faz 🔓 |
|---|---|
| `Pdf.open(caminho)` | abre o PDF pra manipulação em baixo nível |
| `pagina.images.items()` | devolve pares (nome, imagem) de todas as imagens embutidas na página |
| `PdfImage(imagem).extract_to(fileprefix=...)` | salva a imagem extraída em disco, no formato original dela |

> ⚠️ **Atenção:** `pagina.images` está marcado como `deprecated` nas versões recentes do `pikepdf` — ainda funciona, mas o substituto recomendado é `pagina.get_images()`.


In [9]:
import os
from pikepdf import Pdf, PdfImage

arquivo_pikepdf = Pdf.open(caminho_3t20)
pasta_imagens = "../.spec/MGLU_3T20_Imagens"
os.makedirs(pasta_imagens, exist_ok=True)

for pagina in arquivo_pikepdf.pages[:1]:
    for nome, imagem in pagina.images.items():
        imagem_salvar = PdfImage(imagem)
        imagem_salvar.extract_to(fileprefix=f"{pasta_imagens}/{nome}")

print(f"{CinzaClaro}Imagens extraídas:{Reset} {MagentaClaro}{len(os.listdir(pasta_imagens))}{Reset}")


Imagens extraídas: 15


/var/folders/4w/0rjmw_hx1t1fqxbgbqbyn8_h0000gn/T/ipykernel_66654/2378856902.py:9: DeprecationWarning: Page.images is deprecated and will be removed in a future version. Use Page.get_images() instead, which by default also finds images nested inside form XObjects.
  for nome, imagem in pagina.images.items():
/Users/lucaspaguettipereira/Documents/GitHub/Python_Impressionador_HashtagTreinamentos/.venv/lib/python3.14/site-packages/pikepdf/models/image/_classes.py:551: UserWarning: /SMask has a /Matte entry (pre-multiplied alpha) which pikepdf does not undo; colours near edges may be off.
  im = self._apply_mask(im)


## 📝 10. Substituindo Texto num PDF (Nota)

Não é recomendado editar texto dentro de um PDF direto pelo Python — a estrutura interna de um PDF não foi pensada pra isso, e o resultado costuma sair bagunçado (fontes, espaçamento, quebras de linha).

> ⚠️ **Atenção:** se o objetivo é gerar um documento tipo contrato com campos que mudam (nome do cliente, data, valor...), a abordagem certa é montar o texto no **Word** (`python-docx`, como visto na [Mentoria de Python e Word](127-Mentoria%20-%20Python%20e%20Word.ipynb)) e só depois exportar pra PDF — não editar o PDF já pronto.

> 💡 Quem precisar mesmo substituir texto direto num PDF existente, [esse artigo](https://pdf.co/samples/pdf-co-web-api-replace-text-from-pdf-python-replace-text-from-url) mostra uma solução via API que funciona, com alguns cuidados especiais.
